
# Day 16 — SQL for Financial Analytics

## Objective
Learn to query a financial database using SQL and Python.

## Skills
- Inspect database tables
- SELECT and LIMIT
- WHERE and ORDER BY
- GROUP BY and aggregate functions
- Financial reporting with pandas

In [1]:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("hedge_fund.db")

if not db_path.exists():
    raise FileNotFoundError(
        "hedge_fund.db not found. Check your notebook's working directory."
    )

conn = sqlite3.connect(db_path)

print("Connected to:", db_path.resolve())

Connected to: C:\Users\nikit\OneDrive\Documents\AI- Financial -Analyst\hedge_fund.db


In [2]:

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

display(tables)

,name
0,daily_prices


In [3]:

for table_name in tables["name"]:
    print(f"\nTABLE: {table_name}")

    columns = pd.read_sql_query(
        f'PRAGMA table_info("{table_name.replace(chr(34), chr(34)*2)}")',
        conn
    )

    display(columns[["name", "type", "pk"]])


TABLE: daily_prices


,name,type,pk
0,date,TEXT,0
1,ticker,TEXT,0
2,close_price,REAL,0


In [5]:

import sqlite3
import pandas as pd

conn = sqlite3.connect("hedge_fund.db")

# Find the tables in your database
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)

display(tables)

,name
0,daily_prices


In [6]:

for table in tables["name"]:
    print(f"\nTable: {table}")

    columns = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        conn
    )

    display(columns[["name", "type"]])


Table: daily_prices


,name,type
0,date,TEXT
1,ticker,TEXT
2,close_price,REAL


In [7]:

query = """
SELECT *
FROM stock_prices
LIMIT 10;
"""

result = pd.read_sql_query(query, conn)

display(result)

DatabaseError: Execution failed on sql '
SELECT *
FROM stock_prices
LIMIT 10;
': no such table: stock_prices

In [8]:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("hedge_fund.db").resolve()
print("Database:", db_path)
print("File size:", db_path.stat().st_size, "bytes")

conn = sqlite3.connect(db_path)

tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
""", conn)

display(tables)

Database: C:\Users\nikit\OneDrive\Documents\AI- Financial -Analyst\hedge_fund.db
File size: 49152 bytes


,name
0,daily_prices


In [9]:


for table in tables["name"]:
    print(f"\nTABLE: {table}")

    columns = pd.read_sql_query(
        f'PRAGMA table_info("{table}")',
        conn
    )

    display(columns[["name", "type"]])


TABLE: daily_prices


,name,type
0,date,TEXT
1,ticker,TEXT
2,close_price,REAL


In [10]:

query = """
SELECT *
FROM daily_prices
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)
display(df)

,date,ticker,close_price
0,2025-09-23,AAPL,253.493591
1,2025-09-24,AAPL,251.381409
2,2025-09-25,AAPL,255.924637
3,2025-09-26,AAPL,254.519806
4,2025-09-29,AAPL,253.493591
5,2025-09-30,AAPL,253.692871
6,2025-10-01,AAPL,254.509857
7,2025-10-02,AAPL,256.183655
8,2025-10-03,AAPL,257.070374
9,2025-10-06,AAPL,255.745285


In [11]:

query = """
SELECT DISTINCT ticker
FROM daily_prices
ORDER BY ticker;
"""

display(pd.read_sql_query(query, conn))

,ticker
0,AAPL
1,BLK
2,GS
3,JPM
4,MSFT


In [12]:

query = """
SELECT
    ticker,
    COUNT(*) AS trading_days,
    ROUND(AVG(close_price), 2) AS average_price,
    ROUND(MIN(close_price), 2) AS lowest_price,
    ROUND(MAX(close_price), 2) AS highest_price
FROM daily_prices
GROUP BY ticker
ORDER BY ticker;
"""

display(pd.read_sql_query(query, conn))

,ticker,trading_days,average_price,lowest_price,highest_price
0,AAPL,250,281.59,244.37,339.79
1,BLK,250,1057.09,913.04,1177.43
2,GS,250,914.02,730.24,1146.46
3,JPM,250,315.61,280.14,365.18
4,MSFT,250,445.21,352.17,537.65


In [13]:

query = """
SELECT date, ticker, close_price
FROM daily_prices
WHERE date = (
    SELECT MAX(date)
    FROM daily_prices
)
ORDER BY ticker;
"""

display(pd.read_sql_query(query, conn))

,date,ticker,close_price
0,2026-09-21,AAPL,338.980011
1,2026-09-21,BLK,1090.270020
2,2026-09-21,GS,959.390015
3,2026-09-21,JPM,352.040009
4,2026-09-21,MSFT,501.609985


In [14]:

query = """
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT ticker) AS total_stocks,
    MIN(date) AS first_date,
    MAX(date) AS last_date,
    SUM(
        CASE
            WHEN close_price IS NULL
              OR close_price <= 0
            THEN 1
            ELSE 0
        END
    ) AS invalid_prices
FROM daily_prices;
"""

display(pd.read_sql_query(query, conn))

,total_records,total_stocks,first_date,last_date,invalid_prices
0,1250,5,2025-09-23,2026-09-21,0
